# Neural Net Basics — Exercises

**Companion to deck 08 (Tensors + Layers).** PyTorch tensors and the MLP shape.

<a href="https://colab.research.google.com/github/Petkub/MachineLearningLab/blob/main/colab_exercises/07_neural_net.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())

---
## Problem 01 — tensors are NumPy with extras

Build three tensors. Print shape and dtype each.

Tasks:
1. `a` — tensor from list `[[1, 2, 3], [4, 5, 6]]`. Save its shape in `a_shape`.
2. `b` — `(3, 4)` tensor of all zeros, dtype `float32`.
3. `c` — random tensor, shape `(2, 5)`, with `requires_grad=True`.

In [ ]:
# TODO
a = ...
a_shape = ...
b = ...
c = ...

print('a:', a.shape, a.dtype)
print('b:', b.shape, b.dtype)
print('c:', c.shape, 'requires_grad =', c.requires_grad)

In [ ]:
assert tuple(a_shape) == (2, 3), f'a_shape wrong: {a_shape}'
assert b.shape == (3, 4) and b.dtype == torch.float32
assert c.shape == (2, 5) and c.requires_grad
print('Q1 ok')

<details><summary>Hint</summary>

Three constructors:
- `torch.tensor(...)` from a list of lists.
- `torch.zeros(shape, dtype=...)` for filled tensors.
- `torch.randn(shape, requires_grad=...)` for random + tracked.

Shape is an attribute (no parens).

</details>

---
## Problem 02 — vectorized math, NumPy-style

Tensor ops broadcast and vectorize the same way. No loops.

Given `x = torch.arange(10, dtype=torch.float32)`:
1. `y` — `x` squared minus 1, in one expression.
2. `z` — element-wise `max(x, 5)` (clip values below 5 to 5).
3. `dot` — dot product of `x` with itself.

In [ ]:
x = torch.arange(10, dtype=torch.float32)

# TODO
y = ...
z = ...
dot = ...

print('x:', x.tolist())
print('y:', y.tolist())
print('z:', z.tolist())
print('dot:', dot.item())

In [ ]:
assert y.tolist() == [-1, 0, 3, 8, 15, 24, 35, 48, 63, 80]
assert z.tolist() == [5, 5, 5, 5, 5, 5, 6, 7, 8, 9]
assert dot.item() == sum(i * i for i in range(10))
print('Q2 ok')

<details><summary>Hint</summary>

Operators on tensors are element-wise: `x ** 2`, `x - 1`, etc. For element-wise max with a scalar floor, look for a `clamp`-style method or `torch.maximum(a, b)`. Dot product on 1D = `(x * x).sum()` or `x @ x`.

</details>

---
## Problem 03 — move to GPU (or CPU), check shapes

Tasks:
1. Build `device` string — `'cuda'` if available, else `'cpu'`.
2. `t = torch.randn(4, 3)`, then move to `device` and save in `t_dev`.
3. Print: tensor's `.device` attribute. Confirm match.

In [ ]:
# TODO
device = ...
t = torch.randn(4, 3)
t_dev = ...

print('device chosen:', device)
print('t_dev.device:', t_dev.device)

In [ ]:
assert device in ('cuda', 'cpu')
assert str(t_dev.device).startswith(device), f'tensor not on {device}'
print('Q3 ok — on', device)

<details><summary>Hint</summary>

Standard Python conditional for the device string. Tensors expose a `.to(device)` method that returns a copy on the target device. Don't forget to assign back.

</details>

---
## Problem 04 — anatomy of nn.Linear

Tasks:
1. Build `layer = nn.Linear(in_features=4, out_features=2)`.
2. Pass through a batch `x = torch.randn(5, 4)`. Save output in `out`.
3. Save the layer's weight tensor shape in `w_shape`, bias shape in `b_shape`.

In [ ]:
# TODO
layer = ...
x = torch.randn(5, 4)
out = ...
w_shape = ...
b_shape = ...

print('out.shape:', out.shape)
print('weight shape:', w_shape, '| bias shape:', b_shape)

In [ ]:
assert out.shape == (5, 2)
assert tuple(w_shape) == (2, 4), f'weights should be (out, in) = (2, 4), got {tuple(w_shape)}'
assert tuple(b_shape) == (2,)
print('Q4 ok')

<details><summary>Hint</summary>

Calling a layer like a function runs its forward. The weight matrix has shape `(out_features, in_features)` — note the order. Bias has shape `(out_features,)`. Access via `layer.weight.shape` and `layer.bias.shape`.

</details>

---
## Problem 05 — build an MLP with nn.Sequential

Build an MLP that takes 28×28 grayscale images (flattened to 784) and outputs 10 class scores. Use one hidden layer with 64 neurons and ReLU.

Tasks:
1. `model` — `nn.Sequential` with: `Flatten → Linear → ReLU → Linear`.
2. Forward a fake batch `x = torch.randn(8, 1, 28, 28)`. Output shape should be `(8, 10)`.
3. Save total parameter count in `n_params`.

In [ ]:
# TODO
model = ...

x = torch.randn(8, 1, 28, 28)
out = model(x)
n_params = ...

print('out.shape:', out.shape)
print('total params:', n_params)
print(model)

In [ ]:
assert out.shape == (8, 10)
# expected: 784*64 + 64 + 64*10 + 10 = 50_176 + 64 + 640 + 10 = 50_890
assert n_params == 50890, f'expected 50890 params, got {n_params}'
print('Q5 ok')

<details><summary>Hint</summary>

Stack inside `nn.Sequential(layer1, layer2, ...)`. Use `nn.Flatten()` to turn the (B, 1, 28, 28) batch into (B, 784). Then a Linear from 784 → 64, ReLU, Linear 64 → 10. Total parameters: sum `p.numel()` over `model.parameters()`.

</details>

---
## Problem 06 — find the bugs in this MLP

The cell below tries to build an MLP for 28×28 images with 10 output classes. It runs but produces garbage. Find and fix **three bugs**.

Things to look for: missing flatten, mismatched layer sizes, wrong output dim, missing activation.

In [ ]:
# BROKEN — fix three things, save the corrected model in `mlp_q6`.
#
# Goal: input (B, 1, 28, 28) → hidden 128 → 10 output classes.
#
# Bugs:
#  - input shape mismatch (Linear expects (B, 784) but gets (B, 1, 28, 28))
#  - inner sizes don't match between consecutive Linears
#  - missing non-linearity between Linears (just two Linears = one big Linear)

mlp_q6 = nn.Sequential(
    nn.Linear(784, 64),
    nn.Linear(128, 10),
)

# try it
x = torch.randn(4, 1, 28, 28)
try:
    print(mlp_q6(x).shape)
except Exception as e:
    print('ERROR:', e)

In [ ]:
x_test = torch.randn(4, 1, 28, 28)
out = mlp_q6(x_test)
assert out.shape == (4, 10), f'output shape wrong: {out.shape}'
module_types = [type(m).__name__ for m in mlp_q6 if not isinstance(m, nn.Sequential)]
assert 'Flatten' in module_types, 'still missing nn.Flatten() — input must be flattened'
assert any('ReLU' in t or 'GELU' in t or 'Sigmoid' in t or 'Tanh' in t for t in module_types), 'still missing an activation between Linears'
print('Q6 ok — output shape', out.shape)

<details><summary>Hint</summary>

Three failures:
- Image input is 4D `(B, 1, 28, 28)`. Linear wants 2D `(B, 784)`. Add `nn.Flatten()` at the front.
- The first Linear outputs 64; the second Linear's `in_features` must match exactly.
- Two Linears in a row collapse to a single Linear — useless. Insert `nn.ReLU()` between them.

</details>

---
## Solutions

<details><summary>Show all solutions</summary>

```python
# Q1
a = torch.tensor([[1, 2, 3], [4, 5, 6]])
a_shape = a.shape
b = torch.zeros((3, 4), dtype=torch.float32)
c = torch.randn(2, 5, requires_grad=True)

# Q2
y = x ** 2 - 1
z = torch.clamp(x, min=5)
dot = (x * x).sum()   # or x @ x

# Q3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
t_dev = t.to(device)

# Q4
layer = nn.Linear(4, 2)
out = layer(x)
w_shape = layer.weight.shape
b_shape = layer.bias.shape

# Q5
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)
n_params = sum(p.numel() for p in model.parameters())

# Q6 — fixes:
#   - add nn.Flatten() at front
#   - match inner sizes (64 → 64, not 64 → 128)
#   - add nn.ReLU() between Linears
mlp_q6 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)
```
</details>